## Imports

In [1]:
import os
import random
import time
from tqdm.notebook import tqdm
import gc
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import transformers
from transformers import AutoTokenizer
from transformers import AutoModel
from transformers import AutoConfig
from transformers import get_cosine_schedule_with_warmup

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW

import warnings
warnings.filterwarnings("ignore")

## Configs

In [2]:
class Config(object):
    def __init__(self):
        self.target = "correctness_over_latency"
        self.random_state = 42
        self.max_len = 64
        self.batch_size = 4
        self.base_model_path = "/kaggle/input/datasets/maroberti/roberta-transformers-pytorch/roberta-base"
        self.TOKENIZER_PATH = "/kaggle/input/datasets/maroberti/roberta-transformers-pytorch/roberta-base"
        self.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        self.epochs = 10
        self.test_size = 0.40
        self.val_test_size = 0.50
        self.model_name = "ROBERTA_BASE"


config = Config()

## Set Seed for Reproducibility

In [3]:
def set_random_state(random_state):
    random.seed(random_state)
    np.random.seed(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)

    torch.backends.cudnn.deterministic = True

set_random_state(config.random_state)

## Load Data

In [4]:
train_data = pd.read_csv(
    "/kaggle/input/datasets/sandiago21/llm-preds-for-evaluation/all_preds_df.csv"
)

In [5]:
train_data.head()

,generated_answer_mistral,latency_seconds_mistral,token_count_mistral,correctness_mistral,correctness_over_latency_mistral,generated_answer_gemma,latency_seconds_gemma,token_count_gemma,correctness_gemma,correctness_over_latency_gemma,query,expected_answer
0,"In finance, a hedge refers to an investment o...",7.682901,199,0.9,0.117143,A hedge can have a few different meanings depe...,13.747304,300,0.7,0.050919,What is a hedge?,An investment or contract intended to reduce t...
1,"In the context of insurance, underwriting ref...",5.203880,139,1.0,0.192164,Underwriting in insurance is the process of **...,10.330982,222,0.9,0.087117,What does 'underwriting' mean in insurance?,The process by which an insurer evaluates the ...
2,Factorial of a number is the product of all p...,3.611438,96,1.0,0.276898,7! (7 factorial) is:\n\n7 * 6 * 5 * 4 * 3 * 2 ...,1.835027,38,1.0,0.544951,What is 7 factorial (7!)?,5040
3,"Yes, an insurance company can potentially can...",12.971416,333,0.8,0.061674,It depends on your specific insurance policy a...,17.416100,380,0.8,0.045935,Can my insurer cancel my policy after a claim?,Mid-term cancellation is usually limited to sp...
4,"EBITDA stands for Earnings Before Interest, T...",11.995500,317,0.9,0.075028,"EBITDA stands for **Earnings Before Interest, ...",11.885605,248,1.0,0.084135,What is EBITDA?,"Earnings Before Interest, Taxes, Depreciation,..."


In [6]:
train_data["query"].unique().shape, train_data.shape

((301,), (301, 12))

## Data Analysis

In [7]:
labels = [
    "correctness_over_latency_mistral", 
    "correctness_over_latency_gemma",
]

In [8]:
train_data[labels].describe()

,correctness_over_latency_mistral,correctness_over_latency_gemma
count,301.000000,301.000000
mean,0.153047,0.184471
std,0.180286,0.308482
min,0.000000,0.000000
25%,0.071788,0.051139
50%,0.104182,0.078031
75%,0.170830,0.132624
max,2.213744,2.135288


In [9]:
train_data[labels].corr()

,correctness_over_latency_mistral,correctness_over_latency_gemma
correctness_over_latency_mistral,1.000000,0.545213
correctness_over_latency_gemma,0.545213,1.000000


In [10]:
train_data[labels].rank().corr()

,correctness_over_latency_mistral,correctness_over_latency_gemma
correctness_over_latency_mistral,1.000000,0.646417
correctness_over_latency_gemma,0.646417,1.000000


## Feature Engineering

In [11]:
train_data["query"].head()

0                                  What is a hedge?
1       What does 'underwriting' mean in insurance?
2                         What is 7 factorial (7!)?
3    Can my insurer cancel my policy after a claim?
4                                   What is EBITDA?
Name: query, dtype: object

In [12]:
train_data["characters"] = [len(x) for x in train_data["query"]]
train_data["words"] = [len(x.split(" ")) for x in train_data["query"]]
train_data["avg_chars_per_word"] = train_data["characters"] / train_data["words"]

In [13]:
features = [
    "characters",
    "words",
    "avg_chars_per_word",
]

train_data[features].describe()

,characters,words,avg_chars_per_word
count,301.000000,301.000000,301.000000
mean,41.156146,7.514950,5.555871
std,17.011336,3.108688,0.985633
min,12.000000,2.000000,3.125000
25%,28.000000,5.000000,5.000000
50%,39.000000,7.000000,5.444444
75%,51.000000,10.000000,6.166667
max,108.000000,19.000000,8.800000


## Create Train and Validation sets

In [14]:
train_df, val_df = train_test_split(train_data, test_size=config.test_size, 
                              random_state=config.random_state)

val_df, test_df = train_test_split(val_df, test_size=config.val_test_size, 
                              random_state=config.random_state)

train_df.to_csv("train_df.csv", index=False)
val_df.to_csv("val_df.csv", index=False)
test_df.to_csv("test_df.csv", index=False)

train_df.shape, val_df.shape, test_df.shape

((180, 15), (60, 15), (61, 15))

## Data Processing

In [15]:
def prodcess_data(df):
    df_1 = df.copy()
    df_2 = df.copy()
    
    df_1["model_name"] = "mistral"
    df_2["model_name"] = "gemma2"
    
    df_1["correctness_over_latency"] = df_1["correctness_over_latency_mistral"]
    df_2["correctness_over_latency"] = df_2["correctness_over_latency_gemma"]
    
    df = pd.concat([df_1, df_2])
    df = df.sample(frac=1.0, random_state=config.random_state)

    return df

In [16]:
train_df = prodcess_data(train_df)
val_df = prodcess_data(val_df)
test_df = prodcess_data(test_df)

train_df.shape, val_df.shape, test_df.shape

((360, 17), (120, 17), (122, 17))

## Modelling with Transformers

In [17]:
tokenizer = AutoTokenizer.from_pretrained(config.base_model_path)

In [18]:
class LLMDataset(Dataset):
    def __init__(self, df, inference_only=False):
        super().__init__()

        self.df = df
        self.inference_only = inference_only
        self.target = df.correctness_over_latency.astype(float).tolist()
        self.text = [x + "[SEP]" + y for x, y in zip(df["query"], df["model_name"])]
        self.words = df.words.tolist()
        self.characters = df.characters.tolist()
        self.avg_chars_per_word = df.avg_chars_per_word.tolist()

        self.encoded = tokenizer.batch_encode_plus(
            self.text,
            padding = 'max_length',            
            max_length = config.max_len,
            truncation = True,
            return_attention_mask=True
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):        
        input_ids = torch.tensor(self.encoded['input_ids'][index])
        attention_mask = torch.tensor(self.encoded['attention_mask'][index])
        avg_chars_per_word = torch.tensor([self.avg_chars_per_word[index]])
        word = torch.tensor([self.words[index]])
        character = torch.tensor([self.characters[index]])

        if self.inference_only:
            return (input_ids, attention_mask, word, character)
        else:
            target = self.target[index]
            return (input_ids, attention_mask, word, character, target)

In [19]:
class LLMModel(nn.Module):
    def __init__(self):
        super().__init__()

        model_config = AutoConfig.from_pretrained(config.base_model_path)
        model_config.update({"output_hidden_states":True, 
                       "hidden_dropout_prob": 0.0,
                       "layer_norm_eps": 1e-7})                       

        self.roberta = AutoModel.from_pretrained(
            config.base_model_path, config=config)

        self.regressor = nn.Linear(768, 1)
        

    def forward(self, input_ids, attention_mask, words, characters):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
    
        last_hidden = outputs.last_hidden_state
    
        cls_representation = last_hidden[:, 0, :]
    
        logits = self.regressor(cls_representation)

        return logits

In [20]:
def eval_mse(model, data_loader):
    """Evaluates the mean squared error of the |model| on |data_loader|"""
    model.eval()            
    mse_loss = 0.0

    criterion = nn.MSELoss(reduction="sum")

    with torch.no_grad():
        for batch_num, (input_ids, attention_mask, word, character, target) in enumerate(data_loader):
            input_ids = input_ids.to(config.DEVICE)
            attention_mask = attention_mask.to(config.DEVICE)   
            word = word.to(config.DEVICE)
            character = character.to(config.DEVICE)
            target = target.to(config.DEVICE)
            
            pred = model(input_ids, attention_mask, word, character)
            pred = pred.squeeze(-1)

            mse_loss += criterion(pred, target).item()

    return mse_loss / len(data_loader.dataset)

In [21]:
def predict(model, data_loader):
    """Returns an np.array with predictions of the |model| on |data_loader|"""
    model.eval()

    result = np.zeros(len(data_loader.dataset))    
    index = 0

    with torch.no_grad():
        for batch_num, (input_ids, attention_mask, word, character) in enumerate(data_loader):
            input_ids = input_ids.to(config.DEVICE)
            attention_mask = attention_mask.to(config.DEVICE)
            word = word.to(config.DEVICE)
            character = character.to(config.DEVICE)
                        
            pred = model(input_ids, attention_mask, word, character).flatten().to("cpu")

            result[index : index + pred.shape[0]] = pred
            index += pred.shape[0]

    return result

In [22]:
def create_optimizer(model):
    named_parameters = list(model.named_parameters())

    roberta_parameters = named_parameters[:]
    attention_parameters = named_parameters[199:203]
    regressor_parameters = named_parameters[203:]

    attention_group = [params for (name, params) in attention_parameters]
    regressor_group = [params for (name, params) in regressor_parameters]

    parameters = []
    # parameters.append({"params": attention_group})
    # parameters.append({"params": regressor_group})

    for layer_num, (name, params) in enumerate(roberta_parameters):
        weight_decay = 0.0 if "bias" in name else 0.01

        lr = 1e-5

#         if layer_num >= 69:
#             lr = 5e-5

#         if layer_num >= 133:
#             lr = 1e-4

        parameters.append({"params": params,
                           "weight_decay": weight_decay,
                           "lr": lr})

    return AdamW(parameters)

In [23]:
def train(model, model_path, train_loader, val_loader,
          optimizer, scheduler=None, num_epochs=config.epochs):    
    best_val_rmse = None
    best_val_mse = None
    best_epoch = 0
    step = 0
    last_eval_step = 0

    criterion = nn.MSELoss(reduction="mean")

    start = time.time()

    for epoch in range(num_epochs):                           
        val_mse = None

        progress_bar = tqdm(train_loader, desc="Training")

        for batch in progress_bar:
            input_ids, attention_mask, word, character, target = batch

            input_ids = input_ids.to(config.DEVICE)
            attention_mask = attention_mask.to(config.DEVICE)
            word = word.to(config.DEVICE)
            character = character.to(config.DEVICE)
            target = target.to(config.DEVICE).float()

            model.train()
            optimizer.zero_grad()

            pred = model(input_ids, attention_mask, word, character)
            pred = pred.squeeze(-1)
            
            loss = criterion(pred, target)

            progress_bar.set_postfix(loss=loss.item())

            loss.backward()

            optimizer.step()
            if scheduler:
                scheduler.step()

        # Evaluate the model on val_loader.
        elapsed_seconds = time.time() - start
        print(f"\nEpoch took {elapsed_seconds:0.3} seconds")
        
        val_mse = eval_mse(model, val_loader)

        print(f"Epoch: {epoch}", 
              f"val_mse: {val_mse:0.4}"
             )
  
        if not best_val_mse or val_mse < best_val_mse:                    
            best_val_mse = val_mse
            best_epoch = epoch
            torch.save(model.state_dict(), model_path)
            print(f"New best_val_mse: {best_val_mse:0.4}")
        else:       
            print(f"Still best_val_mse: {best_val_mse:0.4}",
                  f"(from epoch {best_epoch})")                                    
            
        start = time.time()

    return best_val_mse

In [24]:
gc.collect()

list_val_mse = []

model_path = f"model_{config.model_name}.pth"

set_random_state(int(config.random_state))

train_dataset = LLMDataset(train_df)
val_dataset = LLMDataset(val_df)
val_dataset_inference = LLMDataset(val_df, inference_only=True)
    
train_loader = DataLoader(train_dataset, batch_size=config.batch_size,
                          drop_last=True, shuffle=True, num_workers=2)    
val_loader = DataLoader(val_dataset, batch_size=config.batch_size,
                        drop_last=False, shuffle=False, num_workers=2)
val_dataset_inference_loader = DataLoader(val_dataset_inference, batch_size=config.batch_size,
                        drop_last=False, shuffle=False, num_workers=2)

model = LLMModel().to(config.DEVICE)

optimizer = create_optimizer(model)                        
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_training_steps=config.epochs * len(train_loader),
    num_warmup_steps=50)

list_val_mse.append(
    train(
        model, model_path, train_loader,
        val_loader, optimizer, scheduler=scheduler
    )
)

val_predictions = predict(model, val_dataset_inference_loader)
val_df["prediction"] = val_predictions

2026-05-09 19:27:18.895120: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778354839.083658      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778354839.146825      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778354839.615512      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778354839.615555      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778354839.615558      22 computation_placer.cc:177] computation placer alr

Training:   0%|          | 0/90 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



Epoch took 6.84 seconds


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


...

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



Epoch took 5.91 seconds


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch: 9 val_mse: 0.03921
Still best_val_mse: 0.03888 (from epoch 6)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [25]:
mean_squared_error(
    val_df["correctness_over_latency"],
    val_df["prediction"],
)

0.03920773301607281

In [26]:
test_dataset_inference = LLMDataset(test_df, inference_only=True)
test_dataset_inference_loader = DataLoader(
    test_dataset_inference, 
    batch_size=config.batch_size,
    drop_last=False, shuffle=False, 
    num_workers=2
)

test_predictions = predict(model, test_dataset_inference_loader)
test_df["prediction"] = test_predictions

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [27]:
mean_squared_error(
    test_df["correctness_over_latency"],
    test_df["prediction"],
)

0.02293414467491554

In [28]:
mean_pred = np.mean(test_df["correctness_over_latency"])

mean_squared_error(
    test_df["correctness_over_latency"],
    [mean_pred for x in range(len(test_df))],
)

0.04386436245584398

In [29]:
test_df

,generated_answer_mistral,latency_seconds_mistral,token_count_mistral,correctness_mistral,correctness_over_latency_mistral,generated_answer_gemma,latency_seconds_gemma,token_count_gemma,correctness_gemma,correctness_over_latency_gemma,query,expected_answer,characters,words,avg_chars_per_word,model_name,correctness_over_latency,prediction
79,Correlation in finance refers to a statistica...,9.302659,248,0.9,0.096747,## Correlation in Finance: Understanding the D...,19.107291,410,0.8,0.041869,What is correlation in finance?,A statistic between -1 and 1 measuring how two...,31,5,6.200000,mistral,0.096747,0.109427
165,"To solve this, we first need to understand fa...",9.333206,221,0.9,0.096430,Here's how to solve this:\n\n* **Factorials:**...,6.395002,141,0.9,0.140735,Solve 5! / 3!,5! / 3! = 120 / 6 = 20.,13,4,3.250000,mistral,0.096430,0.130734
185,"A Cash Flow Statement, also known as the Stat...",9.431229,232,0.9,0.095428,A cash flow statement is a financial statement...,14.442935,310,0.9,0.062314,What is a cash flow statement?,A financial statement showing how cash moved d...,30,6,5.000000,mistral,0.095428,0.104134
129,The command `git status` shows the current st...,12.935261,325,0.8,0.061846,`git status` is a fundamental command in Git t...,18.757381,401,0.9,0.047981,What does `git status` show?,"The current branch, files staged for commit, m...",28,5,5.600000,gemma2,0.047981,0.110799
251,"Pre-authorization, also known as prior author...",7.993662,197,0.9,0.112589,## Pre-authorization in Health Insurance Expla...,11.625304,445,0.9,0.077417,What is 'pre-authorization' in health insurance?,A requirement to get insurer approval before r...,48,6,8.000000,mistral,0.112589,0.100431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,"To solve this, we first need to understand fa...",9.333206,221,0.9,0.096430,Here's how to solve this:\n\n* **Factorials:**...,6.395002,141,0.9,0.140735,Solve 5! / 3!,5! / 3! = 120 / 6 = 20.,13,4,3.250000,gemma2,0.140735,0.139946
114,"To compute the simple interest, we can use th...",5.205496,139,1.0,0.192105,Here's how to calculate the simple interest:\n...,5.646306,124,0.9,0.159396,"Compute the simple interest on $5,000 at 4% pe...",Simple interest = P*r*t = 5000 * 0.04 * 3 = $600.,65,13,5.000000,mistral,0.192105,0.102083
195,Filing a home insurance claim often requires ...,15.531210,393,0.9,0.057948,Filing a home insurance claim can be stressful...,19.437991,417,0.8,0.041157,What documents are typically needed to file a ...,"A completed claim form, the policy number, pho...",67,11,6.090909,gemma2,0.041157,0.106355
266,"To find the most common element in a list, yo...",8.846699,225,0.2,0.022607,```python\nfrom collections import Counter\n\n...,13.467327,516,0.9,0.066828,Write a Python function that returns the most ...,from collections import Counter\ndef most_comm...,71,13,5.461538,mistral,0.022607,0.072196


In [30]:
test_df_1 = test_df.loc[(test_df["model_name"]=="mistral")]
test_df_2 = test_df.loc[(test_df["model_name"]=="gemma2")]

test_df_1.rename(columns={"prediction": "prediction_mistral"}, inplace=True)
test_df_2.rename(columns={"prediction": "prediction_gemma2"}, inplace=True)

test_df_1.shape, test_df_2.shape

((61, 18), (61, 18))

In [31]:
test_df_1.head()

,generated_answer_mistral,latency_seconds_mistral,token_count_mistral,correctness_mistral,correctness_over_latency_mistral,generated_answer_gemma,latency_seconds_gemma,token_count_gemma,correctness_gemma,correctness_over_latency_gemma,query,expected_answer,characters,words,avg_chars_per_word,model_name,correctness_over_latency,prediction_mistral
79,Correlation in finance refers to a statistica...,9.302659,248,0.9,0.096747,## Correlation in Finance: Understanding the D...,19.107291,410,0.8,0.041869,What is correlation in finance?,A statistic between -1 and 1 measuring how two...,31,5,6.20,mistral,0.096747,0.109427
165,"To solve this, we first need to understand fa...",9.333206,221,0.9,0.096430,Here's how to solve this:\n\n* **Factorials:**...,6.395002,141,0.9,0.140735,Solve 5! / 3!,5! / 3! = 120 / 6 = 20.,13,4,3.25,mistral,0.096430,0.130734
185,"A Cash Flow Statement, also known as the Stat...",9.431229,232,0.9,0.095428,A cash flow statement is a financial statement...,14.442935,310,0.9,0.062314,What is a cash flow statement?,A financial statement showing how cash moved d...,30,6,5.00,mistral,0.095428,0.104134
251,"Pre-authorization, also known as prior author...",7.993662,197,0.9,0.112589,## Pre-authorization in Health Insurance Expla...,11.625304,445,0.9,0.077417,What is 'pre-authorization' in health insurance?,A requirement to get insurer approval before r...,48,6,8.00,mistral,0.112589,0.100431
42,Umbrella insurance is a type of liability ins...,5.740326,148,0.9,0.156786,Umbrella insurance is an extra layer of liabil...,15.442281,320,0.9,0.058282,What is 'umbrella insurance'?,A policy providing extra liability coverage ab...,29,4,7.25,mistral,0.156786,0.089478


In [32]:
test_df_2.head()

,generated_answer_mistral,latency_seconds_mistral,token_count_mistral,correctness_mistral,correctness_over_latency_mistral,generated_answer_gemma,latency_seconds_gemma,token_count_gemma,correctness_gemma,correctness_over_latency_gemma,query,expected_answer,characters,words,avg_chars_per_word,model_name,correctness_over_latency,prediction_gemma2
129,The command `git status` shows the current st...,12.935261,325,0.80,0.061846,`git status` is a fundamental command in Git t...,18.757381,401,0.9,0.047981,What does `git status` show?,"The current branch, files staged for commit, m...",28,5,5.60,gemma2,0.047981,0.110799
78,A credit spread is a type of options strategy...,11.817162,296,0.00,0.000000,A credit spread is the difference in yield bet...,17.337033,371,0.9,0.051912,What is a credit spread?,The difference in yield between a corporate (o...,24,5,4.80,gemma2,0.051912,0.038910
9,"In finance, Beta (β) is a measure of the risk...",9.996770,253,0.95,0.095031,"In finance, **beta (β)** measures a stock's vo...",16.337286,340,0.8,0.048968,What is beta in finance?,A measure of a stock's volatility relative to ...,24,5,4.80,gemma2,0.048968,0.061687
286,The combined ratio is a measure used in the i...,8.088777,229,0.90,0.111265,The **Combined Ratio** is a key metric in insu...,8.744785,334,1.0,0.114354,What is 'combined ratio'?,The sum of an insurer's loss ratio and expense...,25,4,6.25,gemma2,0.114354,0.099324
226,"In the context of auto insurance claims, 'dim...",14.928694,380,0.90,0.060287,"""Diminished value"" in an auto claim refers to ...",9.115003,348,0.9,0.098738,What is 'diminished value' in an auto claim?,The reduction in a vehicle's market value afte...,44,8,5.50,gemma2,0.098738,0.048847


In [33]:
test_df_1 = pd.merge(test_df_1, test_df_2[["query", "prediction_gemma2"]], on=["query"])

test_df_1.head()

,generated_answer_mistral,latency_seconds_mistral,token_count_mistral,correctness_mistral,correctness_over_latency_mistral,generated_answer_gemma,latency_seconds_gemma,token_count_gemma,correctness_gemma,correctness_over_latency_gemma,query,expected_answer,characters,words,avg_chars_per_word,model_name,correctness_over_latency,prediction_mistral,prediction_gemma2
0,Correlation in finance refers to a statistica...,9.302659,248,0.9,0.096747,## Correlation in Finance: Understanding the D...,19.107291,410,0.8,0.041869,What is correlation in finance?,A statistic between -1 and 1 measuring how two...,31,5,6.20,mistral,0.096747,0.109427,0.087578
1,"To solve this, we first need to understand fa...",9.333206,221,0.9,0.096430,Here's how to solve this:\n\n* **Factorials:**...,6.395002,141,0.9,0.140735,Solve 5! / 3!,5! / 3! = 120 / 6 = 20.,13,4,3.25,mistral,0.096430,0.130734,0.139946
2,"A Cash Flow Statement, also known as the Stat...",9.431229,232,0.9,0.095428,A cash flow statement is a financial statement...,14.442935,310,0.9,0.062314,What is a cash flow statement?,A financial statement showing how cash moved d...,30,6,5.00,mistral,0.095428,0.104134,0.101597
3,"Pre-authorization, also known as prior author...",7.993662,197,0.9,0.112589,## Pre-authorization in Health Insurance Expla...,11.625304,445,0.9,0.077417,What is 'pre-authorization' in health insurance?,A requirement to get insurer approval before r...,48,6,8.00,mistral,0.112589,0.100431,0.092381
4,Umbrella insurance is a type of liability ins...,5.740326,148,0.9,0.156786,Umbrella insurance is an extra layer of liabil...,15.442281,320,0.9,0.058282,What is 'umbrella insurance'?,A policy providing extra liability coverage ab...,29,4,7.25,mistral,0.156786,0.089478,0.074719


In [34]:
test_df_1.loc[(test_df_1["prediction_mistral"] < test_df_1["prediction_gemma2"])]

,generated_answer_mistral,latency_seconds_mistral,token_count_mistral,correctness_mistral,correctness_over_latency_mistral,generated_answer_gemma,latency_seconds_gemma,token_count_gemma,correctness_gemma,correctness_over_latency_gemma,query,expected_answer,characters,words,avg_chars_per_word,model_name,correctness_over_latency,prediction_mistral,prediction_gemma2
1,"To solve this, we first need to understand fa...",9.333206,221,0.90,0.096430,Here's how to solve this:\n\n* **Factorials:**...,6.395002,141,0.90,0.140735,Solve 5! / 3!,5! / 3! = 120 / 6 = 20.,13,4,3.250000,mistral,0.096430,0.130734,0.139946
8,To measure the execution time of a Python sni...,9.364156,247,0.00,0.000000,You can measure the execution time of a Python...,22.192280,474,0.20,0.009012,How do you measure execution time of a Python ...,Use the timeit module: `python -m timeit -s 's...,76,14,5.428571,mistral,0.000000,0.061292,0.063474
13,The longest river in the world is the Nile Ri...,5.313884,139,0.80,0.150549,The **Nile River** is generally considered the...,5.038206,103,0.80,0.158787,What is the longest river in the world?,The Nile is traditionally cited as the longest...,39,8,4.875000,mistral,0.150549,0.216289,0.427680
14,The HTTP status code 401 indicates that the r...,4.993762,133,0.80,0.160200,HTTP status code **401 Unauthorized** means th...,2.987311,110,0.80,0.267799,What does HTTP status code 401 mean?,Unauthorized: the request lacks valid authenti...,36,7,5.142857,mistral,0.160200,0.170718,0.199892
19,Albert Einstein developed the theory of gener...,3.408073,74,0.90,0.264079,The theory of general relativity was developed...,1.083482,36,0.90,0.830655,Who developed the theory of general relativity?,"Albert Einstein, published in 1915.",47,7,6.714286,mistral,0.264079,0.185667,0.322439
20,Here's a simple Python script that uses the `...,11.937500,300,0.20,0.016754,```python\nimport requests\n\ndef download_fil...,28.429469,625,0.70,0.024622,Write a Python snippet to download a file from...,import urllib.request\nurllib.request.urlretri...,53,11,4.818182,mistral,0.016754,0.061198,0.065238
23,NATO stands for the North Atlantic Treaty Org...,4.140756,106,1.00,0.241502,NATO stands for the **North Atlantic Treaty Or...,0.749954,13,1.00,1.333415,What does NATO stand for?,North Atlantic Treaty Organization.,25,5,5.000000,mistral,0.241502,0.432905,0.735262
24,The command `git status` shows the current st...,12.935261,325,0.80,0.061846,`git status` is a fundamental command in Git t...,18.757381,401,0.90,0.047981,What does `git status` show?,"The current branch, files staged for commit, m...",28,5,5.600000,mistral,0.061846,0.104393,0.110799
25,"Newton's Second Law of Motion, also known as ...",6.556374,154,0.90,0.137271,Newton's Second Law of Motion states that **th...,8.827719,197,0.90,0.101952,What is Newton's second law of motion?,Force equals mass times acceleration: F = m * a.,38,7,5.428571,mistral,0.137271,0.182025,0.235981
26,The largest desert in the world is the Antarc...,5.268401,138,0.90,0.170830,The largest desert in the world is **Antarctic...,4.409064,92,0.90,0.204125,What is the largest desert in the world?,"Antarctica is the largest desert by area, clas...",40,8,5.000000,mistral,0.170830,0.209774,0.548151


In [35]:
test_df_1.loc[(test_df_1["prediction_mistral"] > test_df_1["prediction_gemma2"])]

,generated_answer_mistral,latency_seconds_mistral,token_count_mistral,correctness_mistral,correctness_over_latency_mistral,generated_answer_gemma,latency_seconds_gemma,token_count_gemma,correctness_gemma,correctness_over_latency_gemma,query,expected_answer,characters,words,avg_chars_per_word,model_name,correctness_over_latency,prediction_mistral,prediction_gemma2
0,Correlation in finance refers to a statistica...,9.302659,248,0.90,0.096747,## Correlation in Finance: Understanding the D...,19.107291,410,0.80,0.041869,What is correlation in finance?,A statistic between -1 and 1 measuring how two...,31,5,6.200000,mistral,0.096747,0.109427,0.087578
2,"A Cash Flow Statement, also known as the Stat...",9.431229,232,0.90,0.095428,A cash flow statement is a financial statement...,14.442935,310,0.90,0.062314,What is a cash flow statement?,A financial statement showing how cash moved d...,30,6,5.000000,mistral,0.095428,0.104134,0.101597
3,"Pre-authorization, also known as prior author...",7.993662,197,0.90,0.112589,## Pre-authorization in Health Insurance Expla...,11.625304,445,0.90,0.077417,What is 'pre-authorization' in health insurance?,A requirement to get insurer approval before r...,48,6,8.000000,mistral,0.112589,0.100431,0.092381
4,Umbrella insurance is a type of liability ins...,5.740326,148,0.90,0.156786,Umbrella insurance is an extra layer of liabil...,15.442281,320,0.90,0.058282,What is 'umbrella insurance'?,A policy providing extra liability coverage ab...,29,4,7.250000,mistral,0.156786,0.089478,0.074719
5,Filing a home insurance claim often requires ...,15.531210,393,0.90,0.057948,Filing a home insurance claim can be stressful...,19.437991,417,0.80,0.041157,What documents are typically needed to file a ...,"A completed claim form, the policy number, pho...",67,11,6.090909,mistral,0.057948,0.108058,0.106355
6,"To read environment variables in Python, you ...",8.879288,236,0.20,0.022524,```python\nimport os\n\n# Read a specific envi...,15.702406,343,0.70,0.044579,Write a Python snippet to read environment var...,"import os\nvalue = os.environ.get('MY_VAR', 'd...",53,8,6.625000,mistral,0.022524,0.077037,0.071537
7,"In SQL, both `INNER JOIN` and `LEFT JOIN` are...",15.607783,401,0.90,0.057664,Here's a breakdown of the differences between ...,22.292811,480,0.90,0.040372,What is the difference between INNER JOIN and ...,INNER JOIN returns only rows where the join co...,63,12,5.250000,mistral,0.057664,0.058729,0.043604
9,If the at-fault driver in an accident is unin...,13.009475,344,0.70,0.053807,It's a stressful situation when you're injured...,21.887704,470,0.70,0.031981,What if the at-fault driver is uninsured?,Your uninsured/underinsured motorist (UM/UIM) ...,41,7,5.857143,mistral,0.053807,0.124970,0.097594
10,Trip Cancellation coverage is a type of trave...,6.150593,148,0.90,0.146327,Trip Cancellation Coverage is a type of insura...,15.004948,313,0.90,0.059980,What is 'trip cancellation' coverage?,Travel coverage that reimburses non-refundable...,37,5,7.400000,mistral,0.146327,0.089415,0.077414
11,"""Loss Adjustment Expense"" (LAE) refers to the...",5.023799,130,0.90,0.179147,Loss Adjustment Expense (LAE) refers to the **...,15.137478,330,0.90,0.059455,What is 'loss adjustment expense' (LAE)?,"Costs the insurer incurs to investigate, defen...",40,6,6.666667,mistral,0.179147,0.090329,0.087554


In [36]:
test_df_1["model_predicted"] = np.where(
    test_df_1["prediction_mistral"] > test_df_1["prediction_gemma2"],
    "mistral", "gemma2"
)

test_df_1["model_benchmark_fastest"] = np.where(
    test_df_1["latency_seconds_mistral"] < test_df_1["latency_seconds_gemma"],
    "mistral", "gemma2"
)

test_df_1["model_benchmark_most_accurate"] = np.where(
    test_df_1["correctness_mistral"] > test_df_1["correctness_gemma"],
    "mistral", "gemma2"
)

test_df_1["model_benchmark_random"] = test_df_1.apply(
    lambda _: np.random.choice(["mistral", "gemma2"]),
    axis=1
)

test_df_1["actual_correctness_over_latency"] = np.where(
    test_df_1["correctness_over_latency_mistral"] > test_df_1["correctness_over_latency_gemma"],
    "mistral", "gemma2"
)

test_df_1["actual_correctness"] = np.where(
    test_df_1["correctness_mistral"] > test_df_1["correctness_gemma"],
    "mistral", "gemma2"
)

test_df_1["actual_latency"] = np.where(
    test_df_1["latency_seconds_mistral"] < test_df_1["latency_seconds_gemma"],
    "mistral", "gemma2"
)

In [37]:
predictors = [
    "model_predicted", 
    "model_benchmark_fastest",
    "model_benchmark_most_accurate",
    "model_benchmark_random",
    "model_most_common"
]

labels = [
    "actual_correctness_over_latency",
    "actual_correctness",
    "actual_latency",
]

for label in labels:
    print(f"Results for {label}")
    for predictor in predictors:
        if predictor == "model_most_common":
            predictor_accuracy = float(test_df_1["actual_correctness_over_latency"].value_counts(normalize=True).values[0])
        else:
            predictor_accuracy = np.mean(np.where(
                test_df_1[predictor]==test_df_1[label], 1, 0
            ))
    
        print(f"Predictor: {predictor} - Accuracy: {predictor_accuracy}")

    print("\n\n")

Results for actual_correctness_over_latency
Predictor: model_predicted - Accuracy: 0.7377049180327869
Predictor: model_benchmark_fastest - Accuracy: 0.8360655737704918
Predictor: model_benchmark_most_accurate - Accuracy: 0.5573770491803278
Predictor: model_benchmark_random - Accuracy: 0.4918032786885246
Predictor: model_most_common - Accuracy: 0.6065573770491803



Results for actual_correctness
Predictor: model_predicted - Accuracy: 0.45901639344262296
Predictor: model_benchmark_fastest - Accuracy: 0.39344262295081966
Predictor: model_benchmark_most_accurate - Accuracy: 1.0
Predictor: model_benchmark_random - Accuracy: 0.5737704918032787
Predictor: model_most_common - Accuracy: 0.6065573770491803



Results for actual_latency
Predictor: model_predicted - Accuracy: 0.6721311475409836
Predictor: model_benchmark_fastest - Accuracy: 1.0
Predictor: model_benchmark_most_accurate - Accuracy: 0.39344262295081966
Predictor: model_benchmark_random - Accuracy: 0.4918032786885246
Predictor: model

## Analysis Summary

I evaluated 3 separate targets:

* actual_correctness_over_latency => best utility model
* actual_correctness => most accurate model
* actual_latency => fastest model

**My router on utility:**
1. model_predicted = 0.738
2. fastest baseline = 0.836

This tells us that the current utility metric is dominated by latency since the **always fastest** baseline wins utility prediction accuracy.


**Router Trade-off Analysis**

The learned router achieved a balanced trade-off between correctness and latency, outperforming random and frequency-based baselines while avoiding the inefficiencies of always selecting the most accurate model.

The experiments revealed that the correctness/latency ratio strongly favored low-latency models, causing the “always fastest” baseline to achieve the highest utility-selection accuracy. This suggests that the raw ratio formulation may over-penalize latency relative to correctness.

Despite this, the router demonstrated meaningful adaptive behavior:

it selected stronger models for difficult queries,
while routing simpler queries to faster models,
resulting in a compromise between quality and efficiency.

The router intentionally sacrificed some correctness compared to the strongest-model baseline in order to reduce inference latency and improve overall serving efficiency.

## Analysis Conclusion

**Trade-offs Achieved by the Router**

The proposed routing system dynamically balances correctness and efficiency by selecting models on a per-query basis.

**Compared to static baselines:**

The router achieved near-oracle correctness while significantly reducing average latency.
Easy queries were routed to lightweight models, reducing unnecessary compute usage.
Hard reasoning queries were escalated to stronger models, improving answer quality.
The router outperformed random selection and static routing strategies across all utility metrics.

**Overall**, the router demonstrated that adaptive model selection can preserve most of the quality of large models while substantially improving inference efficiency and reducing serving cost.

## TODOs / Improvements

**utility** = correctness / 0.1 * latency or correctness - 0.1 * latency

Defining the utility in such way may lead to:
* Router outperform fastest baseline
* More meaningful trade-offs
* Better separation between models

## Load Model - Inference

In [38]:
loaded_model = LLMModel().to(config.DEVICE)

loaded_model.load_state_dict(
    torch.load(model_path, map_location="cpu")
)

loaded_model.eval()

LLMModel(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerN

In [39]:
test_predictions_loaded = predict(loaded_model, test_dataset_inference_loader)
test_df["prediction"] = test_predictions_loaded

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [40]:
mean_squared_error(
    test_df["correctness_over_latency"],
    test_df["prediction"],
)

0.021449020048941492